In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from utils import get_dataset


MODEL_ID = "facebook/opt-350m"
model = AutoModelForCausalLM.from_pretrained(
    pretrained_model_name_or_path=MODEL_ID
)
tokenizer = AutoTokenizer.from_pretrained(
    pretrained_model_name_or_path=MODEL_ID,
    clean_up_tokenization_spaces=True,
    padding_side="left"
)

In [2]:
model

OPTForCausalLM(
  (model): OPTModel(
    (decoder): OPTDecoder(
      (embed_tokens): Embedding(50272, 512, padding_idx=1)
      (embed_positions): OPTLearnedPositionalEmbedding(2050, 1024)
      (project_out): Linear(in_features=1024, out_features=512, bias=False)
      (project_in): Linear(in_features=512, out_features=1024, bias=False)
      (layers): ModuleList(
        (0-23): 24 x OPTDecoderLayer(
          (self_attn): OPTAttention(
            (k_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (v_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (q_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (out_proj): Linear(in_features=1024, out_features=1024, bias=True)
          )
          (activation_fn): ReLU()
          (self_attn_layer_norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
          (fc1): Linear(in_features=1024, out_features=4096, bias=True)
          (fc2): Linear(in_features=409

In [3]:
tokenizer.special_tokens_map

{'bos_token': '</s>',
 'eos_token': '</s>',
 'unk_token': '</s>',
 'pad_token': '<pad>'}

In [4]:
esp_dataset = get_dataset(
    data_path="andreamorgar/spanish_poetry",
    split_perc=10
)

fr_dataset = get_dataset(
    data_path="Abirate/french_book_reviews",
    split_perc=10
).rename_column(
    original_column_name="reader_review",
    new_column_name="content"
)

In [5]:
def tokenize_data(data):
    return tokenizer(
        [x for x in data["content"] if x],
        padding="max_length",
        truncation=True,
        max_length=512
    )

esp_tokenized = esp_dataset.map(
    function=tokenize_data,
    batched=True,
    remove_columns=esp_dataset["train"].column_names
)

fr_tokenized = fr_dataset.map(
    function=tokenize_data,
    batched=True,
    remove_columns=fr_dataset["train"].column_names
)

In [6]:
from peft import LoraConfig

lora_config = LoraConfig(
    task_type="CAUSAL_LM",
    r=8,
    lora_alpha=32,
    target_modules=["k_proj", "v_proj", "q_proj", "fc1", "fc2"]
)
lora_config.to_dict()

{'peft_type': <PeftType.LORA: 'LORA'>,
 'auto_mapping': None,
 'base_model_name_or_path': None,
 'revision': None,
 'task_type': 'CAUSAL_LM',
 'inference_mode': False,
 'r': 8,
 'target_modules': {'fc1', 'fc2', 'k_proj', 'q_proj', 'v_proj'},
 'lora_alpha': 32,
 'lora_dropout': 0.0,
 'fan_in_fan_out': False,
 'bias': 'none',
 'use_rslora': False,
 'modules_to_save': None,
 'init_lora_weights': True,
 'layers_to_transform': None,
 'layers_pattern': None,
 'rank_pattern': {},
 'alpha_pattern': {},
 'megatron_config': None,
 'megatron_core': 'megatron.core',
 'loftq_config': {},
 'use_dora': False,
 'layer_replication': None}

In [7]:
model.add_adapter(adapter_config=lora_config, adapter_name="esp_adapter")
model.add_adapter(adapter_config=lora_config, adapter_name="fr_adapter")
model

OPTForCausalLM(
  (model): OPTModel(
    (decoder): OPTDecoder(
      (embed_tokens): Embedding(50272, 512, padding_idx=1)
      (embed_positions): OPTLearnedPositionalEmbedding(2050, 1024)
      (project_out): Linear(in_features=1024, out_features=512, bias=False)
      (project_in): Linear(in_features=512, out_features=1024, bias=False)
      (layers): ModuleList(
        (0-23): 24 x OPTDecoderLayer(
          (self_attn): OPTAttention(
            (k_proj): lora.Linear(
              (base_layer): Linear(in_features=1024, out_features=1024, bias=True)
              (lora_dropout): ModuleDict(
                (esp_adapter): Identity()
                (fr_adapter): Identity()
              )
              (lora_A): ModuleDict(
                (esp_adapter): Linear(in_features=1024, out_features=8, bias=False)
                (fr_adapter): Linear(in_features=1024, out_features=8, bias=False)
              )
              (lora_B): ModuleDict(
                (esp_adapter): Linear(in_f

In [8]:
model.active_adapters()  # the last added adapter

['fr_adapter']

In [9]:
model.set_adapter(adapter_name="esp_adapter")
model.active_adapters()

['esp_adapter']

Another way of adding adapter:

In [10]:
from peft import get_peft_model


peft_model = get_peft_model(
    model=model,
    peft_config=lora_config,
    adapter_name="esp_adapter",
    mixed=False
)

peft_model.add_adapter(adapter_name="fr_adapter", peft_config=lora_config)

print("Active adapter:", peft_model.active_adapters)
peft_model.set_adapter("fr_adapter")
print("Active adapter:", peft_model.active_adapters)

peft_model.print_trainable_parameters()

Active adapter: ['esp_adapter']
Active adapter: ['fr_adapter']
trainable params: 3,145,728 || all params: 337,487,872 || trainable%: 0.9321


In [11]:
from transformers import (
    Trainer,
    TrainingArguments,
    DataCollatorForLanguageModeling
)

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,
    return_tensors="pt"
)

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    learning_rate=2e-5,
    weight_decay=0.01,
    per_device_train_batch_size=128,
    per_device_eval_batch_size=128,
    auto_find_batch_size=True,
    report_to="none"
)

peft_model.set_adapter("esp_adapter")

trainer = Trainer(
    model=peft_model,
    args=training_args,
    train_dataset=esp_tokenized["train"],
    data_collator=data_collator
)

In [12]:
trainer.train()

[2024-09-11 13:26:10,300] [INFO] [real_accelerator.py:203:get_accelerator] Setting ds_accelerator to cuda (auto detect)


/usr/bin/ld: cannot find -lcufile: No such file or directory
collect2: error: ld returned 1 exit status


  0%|          | 0/15 [00:00<?, ?it/s]

  0%|          | 0/27 [00:00<?, ?it/s]

  0%|          | 0/51 [00:00<?, ?it/s]

...

  0%|          | 0/195 [00:00<?, ?it/s]

  0%|          | 0/387 [00:00<?, ?it/s]

  0%|          | 0/771 [00:00<?, ?it/s]

{'loss': 3.3028, 'grad_norm': 5.100916385650635, 'learning_rate': 7.029831387808041e-06, 'epoch': 1.95}
{'train_runtime': 241.0802, 'train_samples_per_second': 6.384, 'train_steps_per_second': 3.198, 'train_loss': 3.277980463050219, 'epoch': 3.0}


TrainOutput(global_step=771, training_loss=3.277980463050219, metrics={'train_runtime': 241.0802, 'train_samples_per_second': 6.384, 'train_steps_per_second': 3.198, 'total_flos': 1463962948337664.0, 'train_loss': 3.277980463050219, 'epoch': 3.0})

In [13]:
peft_model.save_pretrained("results/peft_adapter_examples")

In [31]:
## does not work!
# peft_model.load_adapter(
#     model_id="peft_adapter_examples",
#     adapter_name="esp_adapter"
# )

In [22]:
## does not push LoRA models to hub
# peft_model.push_to_hub(
#     "peft_adapter_examples",
#     safe_serialization=True
# )

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

No files have been modified since last commit. Skipping to prevent empty commit.


CommitInfo(commit_url='https://huggingface.co/gulmert89/peft_adapter_examples/commit/9fe49e1e5f242c9fcd5e1c5e98b582372b8ddaaa', commit_message='Upload model', commit_description='', oid='9fe49e1e5f242c9fcd5e1c5e98b582372b8ddaaa', pr_url=None, pr_revision=None, pr_num=None)

Inference with two adapters:

In [14]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftMixedModel, LoraConfig


MODEL_ID = "facebook/opt-350m"
model = AutoModelForCausalLM.from_pretrained(
    pretrained_model_name_or_path=MODEL_ID
)
tokenizer = AutoTokenizer.from_pretrained(
    pretrained_model_name_or_path=MODEL_ID,
    clean_up_tokenization_spaces=True,
    padding_side="left"
)

lora_config = LoraConfig(
    task_type="CAUSAL_LM",
    r=16,
    lora_alpha=64,
    target_modules=["k_proj", "v_proj", "q_proj", "fc1", "fc2"]
)

peft_model_mixed = PeftMixedModel.from_pretrained(
    model=model,
    model_id="./results/peft_adapter_examples/esp_adapter",
    adapter_name="esp_adapter",
    peft_config=lora_config
).eval()

peft_model_mixed.load_adapter(
    model_id="./results/peft_adapter_examples/esp_adapter",
    adapter_name="fr_adapter"
)

peft_model_mixed.set_adapter(["esp_adapter", "fr_adapter"])
print("Active adapters:", peft_model_mixed.active_adapters)

Active adapters: ['esp_adapter', 'fr_adapter']


In [24]:
prompts = [
    "The capital city of the UK is",
    "La capital de España es",
    "Comment épelez-vous français en français"
]
example_inputs = tokenizer(
    text=prompts,
    return_tensors="pt",
    padding=True
)
# adapter_list = ["__base__", "esp_adapter", "fr_adapter"]
outputs = peft_model_mixed.generate(
    **example_inputs,
    max_new_tokens=20
)
for tokens, prompt in zip(outputs, prompts):
    print(
        "Prompt:",
        prompt,
        "\nAnswer:",
        tokenizer.decode(token_ids=tokens, skip_special_tokens=True)
    )
    print()

Prompt: The capital city of the UK is 
Answer: The capital city of the UK is a place of great beauty, of a great
dance, of a great music, of a

Prompt: La capital de España es 
Answer: La capital de España esa
y el que es el
y el que esa
y el que esa


Prompt: Comment épelez-vous français en français 
Answer: Comment épelez-vous français en français,
y en français, y en français, y en fran



---

In [7]:
from transformers import BitsAndBytesConfig, AutoModelForCausalLM
from peft import prepare_model_for_kbit_training, get_peft_model
import torch


bnb_configs = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True
)

MODEL_ID = "facebook/opt-350m"
model = AutoModelForCausalLM.from_pretrained(
    pretrained_model_name_or_path=MODEL_ID,
    quantization_config=bnb_configs,
    low_cpu_mem_usage=True
)

model = prepare_model_for_kbit_training(model=model)
model = get_peft_model(
    model=model,
    peft_config=lora_config,
    adapter_name="esp_adapter"
)
model

PeftModel(
  (base_model): LoraModel(
    (model): OPTForCausalLM(
      (model): OPTModel(
        (decoder): OPTDecoder(
          (embed_tokens): Embedding(50272, 512, padding_idx=1)
          (embed_positions): OPTLearnedPositionalEmbedding(2050, 1024)
          (project_out): Linear4bit(in_features=1024, out_features=512, bias=False)
          (project_in): Linear4bit(in_features=512, out_features=1024, bias=False)
          (layers): ModuleList(
            (0-23): 24 x OPTDecoderLayer(
              (self_attn): OPTAttention(
                (k_proj): lora.Linear4bit(
                  (base_layer): Linear4bit(in_features=1024, out_features=1024, bias=True)
                  (lora_dropout): ModuleDict(
                    (esp_adapter): Identity()
                  )
                  (lora_A): ModuleDict(
                    (esp_adapter): Linear(in_features=1024, out_features=16, bias=False)
                  )
                  (lora_B): ModuleDict(
                    (esp_a

For more comprehensive path, see my [other notebook](https://colab.research.google.com/drive/1_iYOaheB1g3o72hQDYh3zlX8jtLS2JWV).